# Hall Effect in a Semiconductor — report analysis

Physics 111B, doped germanium, van der Pauw geometry. This notebook generates every plot
and number required by the report section of the manual, from all four data runs.

| § | Report item |
|---|---|
| 1 | Data reduction — how raw voltages become ρ and R_H |
| 2 | **Item 1** — ρ and R_H at room temperature; field where E_Hall = E_ohmic |
| 3 | **Item 2** — ρ vs 1/T; extrinsic and intrinsic regions |
| 4 | **Item 3** — σ vs 1/T, and the band gap |
| 5 | **Item 4** — R_H vs 1/T |
| 6 | **Item 6** — R_H·σ vs T; where R_H → 0 |
| 7 | **Items 5, 10, 11** — mobility ratio, carrier concentration, extrinsic mobilities |
| 8 | **Item 7** — p-type or n-type, and the field-direction question |
| 9 | **Items 8, 9** — carrier concentration and Hall mobility vs T |
| 10 | **Item 12** — magnetoresistance |
| 11 | **Item 13** — comparing runs at different B and different I |
| 12 | Summary table and figure export |

**Your four runs.** Three of them (`neg_1500_100`, `3000_100`, `3000_500`) were taken with
the *multifield* LabView program, which steps the magnet between −B, ~0 and +B on successive
temperature points. Each of those files therefore contains three interleaved runs sharing
one temperature ramp, and gets split into three. `0_100` is a single-state zero-field run.

## 1. Data reduction

### 1.1 What is in the raw file

Each row is one temperature point. For each of four contact configurations the program
sources a current and measures the transverse voltage, then repeats with the current
reversed — the `sample I -AB` / `Voltage -DC` columns. The configurations follow the table
in manual §8.2:

| current | voltage | trans-resistance | used for |
|---|---|---|---|
| I_AB | V_DC | R_AB,DC | resistivity (adjacent contacts) |
| I_AD | V_BC | R_AD,BC | resistivity (adjacent contacts) |
| I_AC | V_BD | R_AC,BD | Hall (diagonal contacts) |
| I_BD | V_AC | R_BD,AC | Hall (diagonal contacts) |

### 1.2 Three averages remove three different systematic errors

**Current reversal** removes thermal EMFs. A thermoelectric voltage at a contact junction
adds to the reading regardless of which way the current flows:

$$V(+I) = RI + V_\text{off},\qquad V(-I) = -RI + V_\text{off}
\;\Longrightarrow\; R = \tfrac12\!\left[\tfrac{V(+I)}{+I} + \tfrac{V(-I)}{-I}\right]$$

Note this is the **sum** of the two quotients. Reversing the current reverses the voltage
too, so both quotients already carry the same sign; their difference is the offset.

**Diagonal averaging** removes contact misalignment. The two Hall contacts are never exactly
opposite each other, so part of the longitudinal (resistive) voltage leaks into the Hall
reading. The two diagonal configurations pick up the same Hall signal but *opposite*
misalignment offsets, so averaging them cancels it — this is why the manual specifies
$(R_{AC,BD} + R_{BD,AC})/2$.

**Field reversal** removes whatever is left. The Hall signal is odd in B, every offset is
even in B:

$$R_\text{Hall} = \tfrac12\left[R_\text{diag}(+B) - R_\text{diag}(-B)\right]$$

§1.5 verifies all three worked.

### 1.3 Van der Pauw resistivity

For an arbitrarily-shaped sample with four edge contacts, ρ solves the transcendental
equation (manual eq. 8)

$$\exp\!\left(\frac{-\pi d}{\rho}R_{AB,DC}\right) + \exp\!\left(\frac{-\pi d}{\rho}R_{AD,BC}\right) = 1$$

which the manual rewrites in closed form (eq. 9) as

$$\rho = \frac{\pi d}{\ln 2}\cdot\frac{R_{AB,DC}+R_{AD,BC}}{2}\cdot f\!\left(\frac{R_{AB,DC}}{R_{AD,BC}}\right),
\qquad f(x) \simeq \frac{1}{\cosh\!\left(\ln(x)/2.403\right)}$$

The geometry factor $f$ corrects for the contacts not being at the corners of a perfect
square. We compute both the exact numerical solution and the approximation, and report the
difference as a check.

### 1.4 Hall coefficient

$$R_H = \frac{R_\text{Hall}\, d}{B} = \frac{V_H\, d}{I\,B}\quad [\mathrm{m^3/C}]$$

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats import linregress
from pathlib import Path

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10, "figure.facecolor": "white"})
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

# ---------------- sample and constants ----------------
d      = 1.25e-3          # thickness, m (manual section 9.1: 1.25 +/- 0.01 mm)
d_err  = 0.01e-3          # thickness uncertainty, m
e      = 1.602176634e-19  # C
k_B    = 1.380649e-23     # J/K
k_eV   = 8.617333262e-5   # eV/K
G2T    = 1e-4             # gauss -> tesla

T_ROOM = 300.0

# ---------------- your run log ----------------
# Filenames are not trusted: 3000_100 actually sat at ~3490 G, and 3000_500 was taken at
# 10 uA despite its name. neg_1500_500 really is 500 uA, and it drops into source
# compliance between ~140 K and ~250 K, where the current sags from 500 to ~190 uA --
# harmless here because every trans-resistance is V/I using the per-point current.
# I_uA and B_G below are for labelling and grouping only; the field that enters R_H is
# always the measured gaussmeter value, point by point.
RUN_INFO = {
    "0_100":        dict(I_uA=10,  B_G=0,    note="zero field, single state"),
    "neg_1500_100": dict(I_uA=100, B_G=1750, note="multifield +/0/-"),
    "3000_100":     dict(I_uA=100, B_G=3500, note="multifield -/0/+"),
    "3000_500":     dict(I_uA=10,  B_G=3600, note="multifield -/0/+, cooled to 77 K"),
    "neg_1500_500": dict(I_uA=500, B_G=1750, note="multifield, low field + HIGH current"),
}

# Established in section 8: the apparatus sign convention is inverted relative to the
# textbook one (which lead the electrometer calls "+" and which way the gaussmeter calls
# B positive are both arbitrary). This factor is applied in section 1.4 so that R_H
# everywhere downstream carries the PHYSICAL sign: positive (hole-like) in the extrinsic
# plateau, negative (electron-like) past T0. The uncorrected values are kept as R_H_raw,
# and section 8 makes the argument from those.
RH_SIGN = -1

CONFIGS = {
    "AB,DC": dict(i="sample I AB", v="Voltage DC", kind="rho"),
    "AD,BC": dict(i="sample I AD", v="Voltage BC", kind="rho"),
    "AC,BD": dict(i="sample I AC", v="Voltage BD", kind="hall"),
    "BD,AC": dict(i="sample I BD", v="Voltage AC", kind="hall"),
}
print("configured")

configured


In [2]:
def trans_R(sub, i, v):
    """Trans-resistance V/I averaged over current reversal -- cancels thermal EMFs."""
    return 0.5 * (sub[f"Voltage {v}"] / sub[f"sample I {i}"]
                  + sub[f"Voltage -{v}"] / sub[f"sample I -{i}"]).to_numpy()


def despike(x, rel_tol=0.15, n_step=6):
    """
    Drop isolated single-point glitches (manual section 9.2, step 16).

    A glitch departs from the straight line through its two neighbours by more than the
    neighbours differ from each other. Comparing to neighbours rather than to a rolling
    median matters because R_Hall genuinely swings through zero near the inversion
    temperature, and a median filter would eat that real structure.
    """
    x = np.asarray(x, float)
    n = len(x)
    if n < 4:
        return x, 0
    mid, spread = np.full(n, np.nan), np.full(n, np.nan)
    mid[1:-1], spread[1:-1] = 0.5 * (x[:-2] + x[2:]), np.abs(x[2:] - x[:-2])
    mid[0], spread[0] = 2 * x[1] - x[2], abs(x[2] - x[1])
    mid[-1], spread[-1] = 2 * x[-2] - x[-3], abs(x[-2] - x[-3])
    dev = np.abs(x - mid)
    step = np.nanmedian(np.abs(np.diff(x[np.isfinite(x)])))
    flag = (dev > np.maximum(n_step * step, rel_tol * np.abs(mid))) & (dev > spread)
    return np.where(flag, np.nan, x), int(flag.sum())


def rho_vdp(R_A, R_B, thickness=d, exact=True):
    """Van der Pauw resistivity. Closed form (manual eq. 9) plus the exact solve of eq. 8."""
    R_A, R_B = np.abs(R_A), np.abs(R_B)
    f = 1.0 / np.cosh(np.log(R_A / R_B) / 2.403)
    approx = np.pi * thickness / np.log(2) * (R_A + R_B) / 2 * f
    if not exact:
        return approx, approx

    def solve(a, b, guess):
        if not (np.isfinite(a) and np.isfinite(b)) or a <= 0 or b <= 0:
            return np.nan
        g = lambda r: np.exp(-np.pi * thickness * a / r) + np.exp(-np.pi * thickness * b / r) - 1
        try:
            return brentq(g, guess * 0.1, guess * 10, xtol=1e-16, rtol=1e-14)
        except Exception:
            return np.nan

    ex = np.array([solve(a, b, g) for a, b, g in zip(R_A, R_B, approx)])
    return np.where(np.isfinite(ex), ex, approx), approx


def load_and_split(path):
    """
    Read one LabView file and split it into field states.

    The multifield program cycles the magnet between -B, ~0 and +B on successive
    temperature points, so one file can hold three interleaved runs. Rows are sorted by
    the gaussmeter VALUE, not by row position, because the cycle order differs between
    files (+0- in one, -0+ in another).
    """
    df = pd.read_csv(path, sep="\t", skiprows=2, engine="python")
    df = df.loc[:, ~df.columns.astype(str).str.match("Unnamed")]
    df.columns = [str(c).strip() for c in df.columns]
    df = df.apply(pd.to_numeric, errors="coerce").dropna(subset=["Temperature (K)"])

    B = df["B-Field (Gauss)"]
    thr = 0.25 * B.abs().max()
    label = np.where(B > thr, "+", np.where(B < -thr, "-", "0"))
    out = {}
    for lab in ("+", "0", "-"):
        sub = df[label == lab].sort_values("Temperature (K)").reset_index(drop=True)
        if len(sub) >= 8:
            out[lab] = sub

    # A single-field file has no cycle to split: the threshold above is relative to the
    # file's own maximum, so a run held at one field lands entirely in one bucket. Relabel
    # it by absolute magnitude instead -- below ~100 G it is a zero-field run (what the
    # gaussmeter reads there is residual hysteretic field, not an applied field).
    if len(out) == 1:
        (lab, sub), = out.items()
        true_lab = "0" if sub["B-Field (Gauss)"].abs().median() < 100 else lab
        out = {true_lab: sub}
    return out


def reduce_state(sub):
    """One field state -> dict of T, rho, sigma, diagonal signal, and diagnostics."""
    R_A, nA = despike(trans_R(sub, "AB", "DC"))
    R_B, nB = despike(trans_R(sub, "AD", "BC"))
    R_1, n1 = despike(trans_R(sub, "AC", "BD"))
    R_2, n2 = despike(trans_R(sub, "BD", "AC"))
    rho, rho_approx = rho_vdp(R_A, R_B)
    return dict(T=sub["Temperature (K)"].to_numpy(),
                B_G=sub["B-Field (Gauss)"].to_numpy(),
                I=np.abs(sub["sample I AB"].to_numpy()),
                R_A=R_A, R_B=R_B,
                R_diag=0.5 * (R_1 + R_2),            # Hall part: misalignment cancels
                R_misalign=0.5 * (R_1 - R_2),        # what that averaging removed
                rho=rho, rho_approx=rho_approx, sigma=1.0 / rho,
                n_cut=nA + nB + n1 + n2)

print("reduction functions defined")

reduction functions defined


In [3]:
# ---- reduce every run, then build one record per (field, current) condition ----
runs, conditions = {}, {}

for name, info in RUN_INFO.items():
    states = {k: reduce_state(v) for k, v in load_and_split(Path("data") / name).items()}
    runs[name] = states

    # temperature grid common to the field states present in this file
    Tg = states["+"]["T"] if "+" in states else states["0"]["T"]

    def og(state, key, Tg=Tg):
        """Interpolate a state onto the common grid (the +B, 0 and -B points of a
        multifield file are taken at different temperatures ~1.5 K apart, because the
        ramp continues between them)."""
        s = states[state]
        m = np.isfinite(s[key])
        return np.interp(Tg, s["T"][m], s[key][m], left=np.nan, right=np.nan)

    rec = dict(file=name, T=Tg, I_uA=info["I_uA"], B_nominal=info["B_G"], note=info["note"],
               rho_zero=og("0", "rho") if "0" in states else None)

    if "+" in states and "-" in states:
        B_T = 0.5 * (np.nanmedian(states["+"]["B_G"]) - np.nanmedian(states["-"]["B_G"])) * G2T
        # Point-by-point field: mean magnitude of the +B and -B gaussmeter readings at each
        # temperature. R_Hall averages the two states, so dividing by their mean magnitude
        # is exact for a signal linear in B -- the +B/-B asymmetry is not an error.
        # (B_T, the run median, is kept for labels only.)
        B_pt = 0.5 * (og("+", "B_G") - og("-", "B_G")) * G2T
        R_hall = 0.5 * (og("+", "R_diag") - og("-", "R_diag"))   # odd in B: the Hall signal
        R_even = 0.5 * (og("+", "R_diag") + og("-", "R_diag"))   # even in B: residual offset
        R_H_raw = R_hall * d / B_pt
        rec.update(B_T=B_T, B_pt=B_pt,
                   rho=0.5 * (og("+", "rho") + og("-", "rho")),
                   R_hall=R_hall, R_even=R_even,
                   R_H_raw=R_H_raw,              # sign as the apparatus reports it
                   R_H=RH_SIGN * R_H_raw,        # physical sign -- justified in section 8
                   has_hall=True)
    else:
        rec.update(B_T=0.0, rho=og("0", "rho"), has_hall=False)

    rec["sigma"] = 1.0 / rec["rho"]
    if rec["has_hall"]:
        rec["mu_hall"] = np.abs(rec["R_H"]) * rec["sigma"]          # |R_H| sigma
        rec["RH_sigma"] = rec["R_H"] * rec["sigma"]                 # signed, for item 6
        rec["carrier"] = 1.0 / (e * np.abs(rec["R_H"]))
    conditions[name] = rec

inventory = pd.DataFrame([
    dict(run=n, states="".join(sorted(runs[n])), I_uA=c["I_uA"],
         B_field_G=round(c["B_T"] / G2T) if c["has_hall"] else 0,
         T_min=round(c["T"].min()), T_max=round(c["T"].max()),
         points=len(c["T"]), Hall=c["has_hall"], note=c["note"])
    for n, c in conditions.items()])
print(inventory.to_string(index=False))

# the reference condition for all single-condition plots: highest current, then highest field
REF = "3000_100"
ref = conditions[REF]
print(f"\nreference condition: {REF}  ({ref['I_uA']:.0f} uA, {ref['B_T']/G2T:.0f} G)")
print("chosen because the manual warns that currents below ~100 uA give voltages too")
print("small to read accurately, and a larger field gives a larger Hall signal.")

         run states  I_uA  B_field_G  T_min  T_max  points  Hall                                 note
       0_100      0    10          0     96    351     168 False             zero field, single state
neg_1500_100    +-0   100       1777     96    346      55  True                     multifield +/0/-
    3000_100    +-0   100       3490     99    349      55  True                     multifield -/0/+
    3000_500    +-0    10       3618     99    349      55  True     multifield -/0/+, cooled to 77 K
neg_1500_500    +-0   500       1784     96    350      56  True multifield, low field + HIGH current

reference condition: 3000_100  (100 uA, 3490 G)
chosen because the manual warns that currents below ~100 uA give voltages too
small to read accurately, and a larger field gives a larger Hall signal.


### 1.5 Verifying the three offset cancellations

Before trusting any number, check that the decomposition behaves the way the physics says it
must. For each field state, split the two diagonal measurements into

$$\text{Hall part} = \tfrac12(R_{AC,BD} + R_{BD,AC}), \qquad
\text{misalignment} = \tfrac12(R_{AC,BD} - R_{BD,AC})$$

The Hall part must be **odd in B** (equal and opposite at ±B, vanishing at zero field). The
misalignment must be **even in B** (the same at all three states, since it is a geometric
property of where the contacts sit, not a magnetic effect). If these came out the other way
round, the contact map or a lead polarity would be wrong and everything downstream would be
meaningless.

In [4]:
rows = []
for name, states in runs.items():
    for lab, s in states.items():
        j = np.argmin(np.abs(s["T"] - T_ROOM))
        rows.append(dict(run=name, state=lab, B_G=round(np.nanmedian(s["B_G"])),
                         hall_part=s["R_diag"][j], misalignment=s["R_misalign"][j],
                         glitches_cut=s["n_cut"]))
check = pd.DataFrame(rows)
print("decomposition of the diagonal measurements at ~300 K (ohms):\n")
print(check.to_string(index=False))

print("\nThe Hall part flips sign with B and is ~0 at zero field; the misalignment is the")
print("same at every field state. Both cancellations work as intended.\n")

r = ref
print(f"residual even-in-B offset in {REF}, as a fraction of the Hall signal at 300 K: "
      f"{abs(np.interp(T_ROOM, r['T'], r['R_even']) / np.interp(T_ROOM, r['T'], r['R_hall']))*100:.2f} %")
for name, c in conditions.items():
    if c["has_hall"]:
        dev = np.nanmax(np.abs(runs[name]["+"]["rho"] - runs[name]["+"]["rho_approx"])
                        / runs[name]["+"]["rho"])
        print(f"exact vs closed-form van der Pauw, {name}: max relative difference {dev:.2e}")

decomposition of the diagonal measurements at ~300 K (ohms):

         run state   B_G  hall_part  misalignment  glitches_cut
       0_100     0    70   0.770000     18.690000             1
neg_1500_100     +  1771  11.353500     21.084000             1
neg_1500_100     0    45   0.302500     19.165500             0
neg_1500_100     - -1782 -10.095250     17.860750             0
    3000_100     +  3482  24.855750     19.334750             0
    3000_100     0   -20  -0.060250     20.091750             0
    3000_100     - -3498 -23.048750     17.438750             1
    3000_500     +  3609  24.802500     19.022500             0
    3000_500     0   -21   0.002500     19.757500             0
    3000_500     - -3627 -22.980000     17.095000             6
neg_1500_500     +  1779   9.943161     16.205209             1
neg_1500_500     0    45   0.329484     19.138922             0
neg_1500_500     - -1790 -10.408818     17.693161             0

The Hall part flips sign with B and is ~0

## 2. Report item 1 — ρ and R_H at room temperature

### The field at which the Hall field equals the ohmic field

Current density $J$ along $x$, field $B$ along $z$. The ohmic field driving the current is
$E_x = \rho J$. The Hall field that builds up transversely is $E_H = R_H J B$. Setting them
equal, the current density cancels:

$$\rho J = R_H J B \;\Longrightarrow\; B = \frac{\rho}{|R_H|} = \frac{1}{\mu_H}$$

So the answer is just the reciprocal of the Hall mobility — a compact statement that mobility
is *the* scale that decides when magnetic deflection starts to compete with ohmic transport.
It is also the condition $\mu_H B \sim 1$, the boundary of the low-field regime that the
simple Hall analysis assumes.

**Which mobility, though?** At 300 K this sample is already past $T_0$ (§6), so $R_H$ is
suppressed by the near-cancellation between electrons and holes, and the $\mu_H$ you measure
there is *not* a single-carrier mobility. The answer "at room temperature" is still what the
question asks for, but it is worth also quoting $1/\mu_p$ from the extrinsic plateau, which
is the field at which a *hole* would be deflected as strongly as it is driven. The two differ
by almost an order of magnitude, and the difference is physics, not error.

### Error propagation

Four terms, evaluated point by point in temperature:

- **Thickness** ($1.25\pm0.01$ mm, 0.80 %) and **instrument gain** (6514 voltage 0.03 %, 2400
  current 0.05 %) are correlated: they are the same for every reading and never average
  down. Because ρ is homogeneous of degree one in $(R_A,R_B)$ and $f$ depends only on
  their ratio, a common gain error passes straight through to ρ. Both cancel in
  $\rho/|R_H| = 1/\mu_H$ (apart from the voltage gain).
- **Temperature.** Near 300 K, ρ changes by ~5 %/K and $R_H$ by ~4 %/K, so a temperature
  error is a large error on the room-temperature values. $\sigma_T$ is measured from the
  data: the zero-field resistivity is a property of the sample alone, so the spread of
  $\rho_0(300\,\mathrm{K})$ across the five runs, divided by $d\ln\rho/dT$, is the
  run-to-run reproducibility of the sample temperature. Each quantity then carries
  $|d\ln y/dT|\,\sigma_T$.
- **Random scatter**, estimated locally from fourth differences
  $y_{i-2}-4y_{i-1}+6y_i-4y_{i+1}+y_{i+2}$, which cancel any trend up to a cubic.
  (Second differences do not work here: the curvature of ρ(T) over one 4.8 K step near
  300 K is ~4 %, far larger than the noise.)
- **Gaussmeter calibration** enters $R_H$ only. Set `GAUSS_CAL_REL` from its spec.

The field used in $R_H$ is the measured gaussmeter value at each temperature (mean of the
$+B$ and $-B$ magnitudes). The $\pm B$ asymmetry is not an error, because $R_\text{Hall}$
averages the two states.


In [5]:
SIG_V_REL, SIG_I_REL = 3e-4, 5e-4          # 6514 voltage and 2400 current reading accuracy (gain)
gain = np.hypot(SIG_V_REL, SIG_I_REL)
# Gaussmeter calibration accuracy (relative). Set this from the gaussmeter spec in the
# manual; it is a correlated error on R_H only. None = not included (flagged in output).
GAUSS_CAL_REL = None


def scatter(y):
    """Robust random scatter of a series, from second differences (trend-insensitive).
    Kept for the despike/diagnostic code; NOT used for the error budget any more, because
    near 300 K the curvature of rho(T) over one 4.8 K step (~4 %) swamps the noise."""
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    dev = y[1:-1] - 0.5 * (y[:-2] + y[2:])
    return np.median(np.abs(dev)) / 0.6745 / np.sqrt(1.5)


def local_noise(y, half=5):
    """
    Local random scatter of a series at each point, from FOURTH differences
    y[i-2] - 4y[i-1] + 6y[i] - 4y[i+1] + y[i+2]. For white noise of std s this has std
    s*sqrt(70); any smooth trend up to a cubic cancels exactly, so the steep curvature
    of rho(T) and R_H(T) near room temperature does not leak in (it does with second
    differences). Robust median over a +/- `half`-point window, so it follows how the
    noise changes with temperature instead of applying one run-wide number everywhere.
    """
    y = np.asarray(y, float)
    d4 = np.full_like(y, np.nan)
    d4[2:-2] = y[:-4] - 4 * y[1:-3] + 6 * y[2:-2] - 4 * y[3:-1] + y[4:]
    s = pd.Series(np.abs(d4)).rolling(2 * half + 1, center=True, min_periods=4).median()
    return (s / 0.6745 / np.sqrt(70)).bfill().ffill().to_numpy()


def dlog_dT(T, y):
    """Local logarithmic temperature sensitivity d ln|y| / dT."""
    return np.gradient(np.log(np.abs(y)), T)


# ---- temperature reproducibility, measured from the data ----
# Near 300 K, rho changes by ~5 %/K and R_H by ~4 %/K, so a temperature error is a large
# error on the room-temperature values. Every run has a zero-field state, and the
# zero-field resistivity is a property of the sample alone, so the spread of rho_0(300 K)
# across the five runs measures how reproducibly the thermometer reads the sample
# temperature (sensor offset, lag on the 2 K/min ramp, run-to-run drift).
def at_T(c, key, T0=T_ROOM):
    y = c[key] if isinstance(key, str) else key
    m = np.isfinite(y)
    return float(np.interp(T0, c["T"][m], np.asarray(y)[m]))


rho0_runs = {n: at_T(c, c["rho_zero"] if c["rho_zero"] is not None else "rho")
             for n, c in conditions.items()}
_r = np.array(list(rho0_runs.values()))
_ref = conditions[REF]
SENS_RHO_300 = abs(at_T(_ref, dlog_dT(_ref["T"], _ref["rho"])))
SIG_T = (np.std(_r, ddof=1) / np.mean(_r)) / SENS_RHO_300     # K


def uncertainties(name):
    """Full error budget for rho, R_H (and their ratio) of one condition, as arrays over T."""
    c = conditions[name]
    T, rho = c["T"], c["rho"]

    rnd_rho = rho * local_noise(np.log(rho))                    # random, from the data
    tmp_rho = rho * np.abs(dlog_dT(T, rho)) * SIG_T             # temperature reproducibility
    sig_rho = np.sqrt(rnd_rho**2 + tmp_rho**2 + (rho * gain)**2 + (rho * d_err / d)**2)
    out = dict(sig_rho=sig_rho, rnd_rho=rnd_rho, tmp_rho=tmp_rho)

    if c["has_hall"]:
        RH = c["R_H"]
        aRH = np.abs(RH)
        rnd_RH = local_noise(RH)
        tmp_RH = aRH * np.abs(dlog_dT(T, RH)) * SIG_T
        cal = aRH * (GAUSS_CAL_REL or 0.0)
        sig_RH = np.sqrt(rnd_RH**2 + tmp_RH**2 + (aRH * gain)**2
                         + (aRH * d_err / d)**2 + cal**2)

        # rho/|R_H| = 1/mu_H: thickness and current gain cancel; the temperature error is
        # common to both, so it enters through d ln(rho/|R_H|)/dT; random terms add.
        ratio = rho / aRH
        rnd_ratio = ratio * np.hypot(rnd_rho / rho, rnd_RH / aRH)
        tmp_ratio = ratio * np.abs(dlog_dT(T, ratio)) * SIG_T
        sig_ratio = np.sqrt(rnd_ratio**2 + tmp_ratio**2
                            + (ratio * SIG_V_REL)**2 + (ratio * (GAUSS_CAL_REL or 0.0))**2)
        out.update(sig_RH=sig_RH, rnd_RH=rnd_RH, tmp_RH=tmp_RH,
                   ratio=ratio, sig_ratio=sig_ratio, rnd_ratio=rnd_ratio, tmp_ratio=tmp_ratio)
    return out


err = {n: uncertainties(n) for n in conditions}

rows = []
for name, c in conditions.items():
    if not c["has_hall"]:
        continue
    E = err[name]
    rho0, RH0 = at_T(c, "rho"), at_T(c, "R_H")
    Beq, sBeq = at_T(c, E["ratio"]), at_T(c, E["sig_ratio"])
    rows.append(dict(run=name, I_uA=c["I_uA"], B_G=round(c["B_T"] / G2T),
                     rho=rho0, sig_rho=at_T(c, E["sig_rho"]),
                     rho_zero=at_T(c, "rho_zero") if c["rho_zero"] is not None else np.nan,
                     R_H=RH0, sig_RH=at_T(c, E["sig_RH"]),
                     mu_H=1 / Beq, sig_mu=sBeq / Beq**2, B_equal_T=Beq, sig_B_equal=sBeq))
room = pd.DataFrame(rows)

print(f"Room temperature ({T_ROOM:.0f} K) values\n")
for _, r in room.iterrows():
    print(f"{r['run']}  ({r['I_uA']:.0f} uA, {r['B_G']:.0f} G)")
    print(f"   rho(B)   = {r['rho']:.4f} +/- {r['sig_rho']:.4f} Ohm m  ({100*r['sig_rho']/r['rho']:.2f} %)")
    print(f"   rho(B=0) = {r['rho_zero']:.4f} Ohm m   (magnetoresistance {100*(r['rho']/r['rho_zero']-1):+.2f} %)")
    print(f"   sigma    = 1/rho(B) = {1/r['rho']:.4f} +/- {r['sig_rho']/r['rho']**2:.4f} S/m")
    print(f"   R_H      = {r['R_H']:+.4f} +/- {r['sig_RH']:.4f} m^3/C   ({100*r['sig_RH']/abs(r['R_H']):.2f} %)")
    print(f"   mu_H     = {r['mu_H']*1e4:.0f} +/- {r['sig_mu']*1e4:.0f} cm^2/Vs")
    print(f"   E_Hall = E_ohmic at B = rho/|R_H| = {r['B_equal_T']:.2f} +/- {r['sig_B_equal']:.2f} T\n")

c = ref
mu_ext = np.nanmedian(np.abs(c["R_H"])[(c["T"] > 110) & (c["T"] < 200)]) * \
         np.nanmedian(c["sigma"][(c["T"] > 110) & (c["T"] < 200)])
print(f"for contrast, using the single-carrier extrinsic mobility of {REF}:")
print(f"   mu_p (110-200 K) = {mu_ext*1e4:.0f} cm^2/Vs  ->  B = 1/mu_p = {1/mu_ext:.2f} T\n")

E = err[REF]
rho0, RH0, Beq = at_T(c, "rho"), abs(at_T(c, "R_H")), at_T(c, E["ratio"])
pct = lambda a, b: 100 * a / b
print(f"temperature reproducibility: zero-field rho(300 K) across the five runs =")
for n, v in rho0_runs.items():
    print(f"      {n:<14s} {v:.4f} Ohm m")
print(f"   spread {100*np.std(_r, ddof=1)/np.mean(_r):.2f} %, d ln rho/dT = {100*SENS_RHO_300:.2f} %/K"
      f"  ->  sigma_T = {SIG_T:.2f} K\n")
print(f"error budget for {REF} at 300 K (relative, %):")
print(f"   {'source':<32s}{'rho':>7s}{'R_H':>7s}{'rho/|R_H|':>11s}")
print(f"   {'thickness':<32s}{pct(d_err,d):7.2f}{pct(d_err,d):7.2f}{'cancels':>11s}")
print(f"   {'instrument gain':<32s}{gain*100:7.2f}{gain*100:7.2f}{SIG_V_REL*100:11.2f}")
print(f"   {'temperature (sigma_T=%.2f K)' % SIG_T:<32s}{pct(at_T(c,E['tmp_rho']),rho0):7.2f}"
      f"{pct(at_T(c,E['tmp_RH']),RH0):7.2f}{pct(at_T(c,E['tmp_ratio']),Beq):11.2f}")
print(f"   {'random (local 4th differences)':<32s}{pct(at_T(c,E['rnd_rho']),rho0):7.2f}"
      f"{pct(at_T(c,E['rnd_RH']),RH0):7.2f}{pct(at_T(c,E['rnd_ratio']),Beq):11.2f}")
print(f"   {'total':<32s}{pct(at_T(c,E['sig_rho']),rho0):7.2f}"
      f"{pct(at_T(c,E['sig_RH']),RH0):7.2f}{pct(at_T(c,E['sig_ratio']),Beq):11.2f}")
if GAUSS_CAL_REL is None:
    print("\n   NOTE: gaussmeter calibration NOT included -- set GAUSS_CAL_REL from its spec.")


Room temperature (300 K) values

neg_1500_100  (100 uA, 1777 G)
   rho(B)   = 0.7334 +/- 0.0089 Ohm m  (1.21 %)
   rho(B=0) = 0.7284 Ohm m   (magnetoresistance +0.69 %)
   sigma    = 1/rho(B) = 1.3635 +/- 0.0166 S/m
   R_H      = -0.0743 +/- 0.0009 m^3/C   (1.17 %)
   mu_H     = 1015 +/- 6 cm^2/Vs
   E_Hall = E_ohmic at B = rho/|R_H| = 9.85 +/- 0.06 T

3000_100  (100 uA, 3490 G)
   rho(B)   = 0.7459 +/- 0.0090 Ohm m  (1.21 %)
   rho(B=0) = 0.7200 Ohm m   (magnetoresistance +3.60 %)
   sigma    = 1/rho(B) = 1.3407 +/- 0.0162 S/m
   R_H      = -0.0901 +/- 0.0010 m^3/C   (1.16 %)
   mu_H     = 1209 +/- 5 cm^2/Vs
   E_Hall = E_ohmic at B = rho/|R_H| = 8.27 +/- 0.03 T

3000_500  (10 uA, 3618 G)
   rho(B)   = 0.7458 +/- 0.0090 Ohm m  (1.21 %)
   rho(B=0) = 0.7199 Ohm m   (magnetoresistance +3.59 %)
   sigma    = 1/rho(B) = 1.3409 +/- 0.0162 S/m
   R_H      = -0.0872 +/- 0.0010 m^3/C   (1.13 %)
   mu_H     = 1169 +/- 4 cm^2/Vs
   E_Hall = E_ohmic at B = rho/|R_H| = 8.55 +/- 0.03 T

neg_1500_5

In [6]:
c = ref
E = err[REF]
# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

m = np.isfinite(c["rho"])
ax[0].errorbar(c["T"][m], c["rho"][m], yerr=E["sig_rho"][m], fmt=".", ms=4,
               elinewidth=0.6, capsize=0, color="tab:blue")
ax[0].axvline(T_ROOM, color="crimson", ls="--", lw=1)
ax[0].plot([T_ROOM], [at_T(c, "rho")], "o", color="crimson", ms=7, zorder=5)
ax[0].annotate(f"{at_T(c,'rho'):.3f} $\\Omega$m", (T_ROOM, at_T(c, "rho")),
               xytext=(5, 50), textcoords="offset points", color="crimson")
ax[0].set(xlabel="T (K)", ylabel=r"$\rho$ ($\Omega\,$m)", title="Resistivity")

m = np.isfinite(c["R_H"])
ax[1].errorbar(c["T"][m], c["R_H"][m], yerr=E["sig_RH"][m], fmt=".", ms=4,
               elinewidth=0.6, capsize=0, color="tab:blue")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].axvline(T_ROOM, color="crimson", ls="--", lw=1)
ax[1].plot([T_ROOM], [at_T(c, "R_H")], "o", color="crimson", ms=7, zorder=5)
ax[1].annotate(f"{at_T(c,'R_H'):+.4f} m$^3$/C", (T_ROOM, at_T(c, "R_H")),
               xytext=(0, -50), textcoords="offset points", color="crimson")
ax[1].set(xlabel="T (K)", ylabel=r"$R_H$ (m$^3$/C)",
          title="Hall coefficient")

for f, nm in zip(figs, ["item1_room_temperature_rho", "item1_room_temperature_RH"]):
    f.suptitle(f"{REF}: {c['I_uA']:.0f} $\\mu$A, {c['B_T']/G2T:.0f} G", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()


## 3. Report item 2 — ρ versus inverse temperature

### What the plot shows and why 1/T

Plotting against $1/T$ is the convention because thermally activated processes are straight
lines on it. In the intrinsic region the carrier density carries a Boltzmann factor
$n_i \propto \exp(-E_g/2k_BT)$, so $\ln\sigma$ falls linearly in $1/T$ with slope
$-E_g/2k_B$ — a straight line whose slope *is* the band gap.

### Identifying the two regions

The two regions have opposite physics, and ρ behaves oppositely in each:

- **Extrinsic (low T).** Every acceptor is ionised, so the hole density is pinned at $N_A$
  and does not change with temperature. The only temperature-dependent quantity left is the
  mobility, which falls as lattice vibrations increase ($\mu_p \propto T^{-\alpha}$). So ρ
  *rises* with temperature — the sample behaves like a metal.
- **Intrinsic (high T).** Electron–hole pairs are generated exponentially fast. The carrier
  density now swamps the mobility decrease, so ρ *collapses*.

The boundary is therefore the **maximum of ρ**, equivalently where $d\sigma/dT = 0$. The
right panel makes this quantitative by differentiating: the local slope of $\ln\sigma$ vs
$1/T$, expressed as an apparent gap $E_\text{app} = -2k_B\,d\ln\sigma/d(1/T)$, is near zero
while extrinsic and climbs to the true gap once fully intrinsic. The temperature where it
leaves zero and the temperature where it reaches the plateau bracket a crossover region
rather than a sharp boundary.

In [7]:
def apparent_gap(T, sigma, window=5):
    """Local slope of ln(sigma) vs 1/T, expressed in eV as -2k dln(sigma)/d(1/T)."""
    m = np.isfinite(sigma)
    Tm, sm = T[m], np.log(sigma[m])
    sm = pd.Series(sm).rolling(window, center=True, min_periods=2).mean().to_numpy()
    return Tm, -2 * k_eV * np.gradient(sm, 1 / Tm)


def first_crossing(x, y, level, rising=True):
    """First temperature at which y crosses `level`. np.interp cannot be used here:
    the apparent-gap curve rises and then falls slightly, so it is not monotonic and
    np.interp would silently return the LAST crossing."""
    y = np.asarray(y, float)
    s = np.sign(y - level)
    for i in range(1, len(s)):
        if s[i] != s[i - 1] and (s[i] > 0) == rising:
            return x[i - 1] + (x[i] - x[i - 1]) * (level - y[i - 1]) / (y[i] - y[i - 1])
    return np.nan


c = ref
T_gap, E_app = apparent_gap(c["T"], c["sigma"])
T_rho_max = c["T"][np.nanargmax(c["rho"])]

# where the apparent gap leaves zero, and where it first reaches the Ge gap
T_leave = first_crossing(T_gap, E_app, 0.05)
T_full = first_crossing(T_gap, E_app, 0.67)

# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

m = np.isfinite(c["rho"])
ax[0].plot(1000 / c["T"][m], c["rho"][m], ".", ms=4)
ax[0].axvline(1000 / T_rho_max, color="crimson", ls="--", lw=1.2)
ax[0].annotate(f"$\\rho$ max = {T_rho_max:.0f} K", (1000 / T_rho_max, np.nanmax(c["rho"])),
               xytext=(16, -6), textcoords="offset points", color="crimson", fontsize=9)
# NOTE: on a 1/T axis, high temperature is on the LEFT -- so the intrinsic region
# (high T) shades the left-hand side and the extrinsic region the right.
ax[0].axvspan(1000 / c["T"].max(), 1000 / T_full, color="tab:orange", alpha=0.12)
ax[0].axvspan(1000 / T_leave, 1000 / c["T"].min(), color="tab:blue", alpha=0.12)
ax[0].text(0.04, 0.55, "intrinsic", transform=ax[0].transAxes, color="tab:orange",
           fontweight="bold", rotation=90, va="center")
ax[0].text(0.70, 0.88, "extrinsic", transform=ax[0].transAxes, color="tab:blue",
           fontweight="bold")
ax[0].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$\rho$ ($\Omega\,$m)")
ax[0].set_title(r"$\rho$ vs inverse temperature", pad=28)
sec = ax[0].secondary_xaxis("top", functions=(lambda x: 1000 / np.where(x == 0, np.nan, x),
                                              lambda t: 1000 / np.where(t == 0, np.nan, t)))
sec.set_xlabel("T (K)")

ax[1].plot(T_gap, E_app, "-", lw=1.5)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].axhline(0.67, color="crimson", ls="--", lw=1)
ax[1].annotate("Ge gap 0.67 eV", (T_gap.min(), 0.67), xytext=(6, 6),
               textcoords="offset points", color="crimson", fontsize=8)
ax[1].axvspan(T_leave, T_full, color="0.6", alpha=0.2)
ax[1].annotate(f"crossover\n{T_leave:.0f}–{T_full:.0f} K", ((T_leave + T_full) / 2, 0.3),
               ha="center", fontsize=9, color="0.25")
ax[1].set(xlabel="T (K)", ylabel=r"$-2k_B\,d\ln\sigma/d(1/T)$  (eV)",
          title="Local slope: apparent gap")

for f, nm in zip(figs, ["item2_rho_vs_invT", "item2_apparent_gap"]):
    f.suptitle("Extrinsic and intrinsic regions", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

print(f"rho maximum (d(sigma)/dT = 0)          : {T_rho_max:.0f} K")
print(f"apparent gap leaves zero               : {T_leave:.0f} K")
print(f"apparent gap reaches the Ge gap        : {T_full:.0f} K")
print(f"\n  extrinsic region : T < {T_leave:.0f} K")
print(f"  crossover        : {T_leave:.0f}-{T_full:.0f} K")
print(f"  intrinsic region : T > {T_full:.0f} K")
print(f"\nQuote {T_rho_max:.0f} K as the transition if a single number is wanted -- it is the")
print("one unambiguous, definition-free marker (the maximum of the measured curve).")

rho maximum (d(sigma)/dT = 0)          : 242 K
apparent gap leaves zero               : 245 K
apparent gap reaches the Ge gap        : 271 K

  extrinsic region : T < 245 K
  crossover        : 245-271 K
  intrinsic region : T > 271 K

Quote 242 K as the transition if a single number is wanted -- it is the
one unambiguous, definition-free marker (the maximum of the measured curve).


## 4. Report item 3 — conductivity versus inverse temperature, and the band gap

Same data as item 2, inverted and plotted logarithmically. This is the form in which the
intrinsic region becomes a straight line whose slope gives the band gap:

$$\sigma \simeq \sigma_0 \exp\!\left(-\frac{E_g}{2k_BT}\right)
\;\Longrightarrow\; \ln\sigma = \text{const} - \frac{E_g}{2k_B}\cdot\frac{1}{T}$$

The factor of 2 is because creating one electron–hole *pair* costs $E_g$, but the Fermi level
sits mid-gap, so each carrier species is activated with $E_g/2$.

### Two corrections you should make before comparing to a textbook number

**1. The mobility's temperature dependence.** $\sigma = n_i e(\mu_n+\mu_p)$ contains the
mobility as well as the carrier density. With $n_i \propto T^{3/2}e^{-E_g/2k_BT}$ and
$\mu \propto T^{-\alpha}$,

$$\sigma \propto T^{3/2-\alpha}\exp\!\left(-\frac{E_g}{2k_BT}\right)$$

The textbook shortcut assumes $\alpha = 3/2$ exactly, so the two powers cancel and the plain
$\ln\sigma$ slope is clean. Your extrinsic region measures $\alpha$ directly (item 11), and
it is not exactly 3/2 — so fit $\ln(\sigma\,T^{\alpha - 3/2})$ instead, using your own
measured $\alpha$.

**2. The gap itself depends on temperature — and this is the big one.** Germanium's gap
narrows as the lattice expands, following the Varshni form

$$E_g(T) = E_g(0) - \frac{4.774\times10^{-4}\,T^2}{T+235}\ \text{eV}, \qquad E_g(0) = 0.744\ \text{eV}$$

which gives 0.66 eV at 300 K — the number usually quoted for germanium. But writing
$E_g(T) \approx E_g(0) - \gamma T$ near the fit window,

$$\exp\!\left(-\frac{E_g(T)}{2k_BT}\right) = \underbrace{\exp\!\left(\frac{\gamma}{2k_B}\right)}_{\text{constant prefactor}}\cdot\exp\!\left(-\frac{E_g(0)}{2k_BT}\right)$$

the linear part is absorbed into the *intercept*, not the slope. **An Arrhenius fit therefore
returns $E_g(0) \approx 0.744$ eV, not the room-temperature 0.66 eV.** Comparing your slope
against 0.66 eV and reporting a 15 % discrepancy is the most common error in this lab.

In [8]:
EG_0K = 0.744        # Varshni Eg(0) for germanium, eV -- what an Arrhenius slope returns
varshni = lambda T: EG_0K - 4.774e-4 * T ** 2 / (T + 235)


def fit_gap(T, sigma, window, power=0.0):
    """
    Fit ln(sigma * T^power) vs 1/T over a window; slope -> E_g.

    power = 0                : the textbook shortcut, assumes mu ~ T^-3/2 exactly
    power = alpha - 3/2      : uses your own measured mobility exponent
    """
    m = np.isfinite(sigma) & (T >= window[0]) & (T <= window[1]) & (sigma > 0)
    x = 1 / T[m]
    y = np.log(sigma[m] * T[m] ** power)
    r = linregress(x, y)
    return dict(Eg=-2 * r.slope * k_eV, err=2 * r.stderr * k_eV, r2=r.rvalue ** 2,
                slope=r.slope, intercept=r.intercept, n=int(m.sum()), x=x, y=y)


INTRINSIC = (300.0, 350.0)       # set from the apparent-gap plateau in item 2
c = ref

# measured mobility exponent from the extrinsic region (sigma = e N_A mu_p there,
# so the sigma power law IS the mobility power law)
m_ext = (c["T"] >= 110) & (c["T"] <= 200) & np.isfinite(c["sigma"])
ALPHA = -np.polyfit(np.log(c["T"][m_ext]), np.log(c["sigma"][m_ext]), 1)[0]

plain = fit_gap(c["T"], c["sigma"], INTRINSIC, power=0.0)
corr = fit_gap(c["T"], c["sigma"], INTRINSIC, power=ALPHA - 1.5)

# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

m = np.isfinite(c["sigma"])
ax[0].semilogy(1000 / c["T"][m], c["sigma"][m], ".", ms=4)
ax[0].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$\sigma$ (S/m)")
ax[0].set_title(r"$\sigma$ vs inverse temperature", pad=28)
sec = ax[0].secondary_xaxis("top", functions=(lambda x: 1000 / np.where(x == 0, np.nan, x),
                                              lambda t: 1000 / np.where(t == 0, np.nan, t)))
sec.set_xlabel("T (K)")

ax[1].plot(1000 / c["T"][m], np.log(c["sigma"][m]), ".", ms=4, color="0.65", label="all data")
ax[1].plot(1000 * plain["x"], plain["y"], ".", ms=6, color="tab:blue", label="intrinsic window")
ax[1].plot(1000 * plain["x"], plain["intercept"] + plain["slope"] * plain["x"], "-",
           color="crimson", lw=1.6, label=f"fit: $E_g$ = {plain['Eg']:.3f} eV")
ax[1].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$\ln\sigma$", title="Band gap fit")
ax[1].legend(fontsize=8)

for f, nm in zip(figs, ["item3_sigma_vs_invT", "item3_band_gap_fit"]):
    f.suptitle("Conductivity and the band gap", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

print(f"intrinsic fit window: {INTRINSIC[0]:.0f}-{INTRINSIC[1]:.0f} K")
print(f"measured mobility exponent from the extrinsic region: alpha = {ALPHA:.2f}\n")
print(f"  plain  ln(sigma)          vs 1/T : Eg = {plain['Eg']:.3f} +/- {plain['err']:.3f} eV  (r^2 = {plain['r2']:.5f})")
print(f"  corr.  ln(sigma T^{ALPHA-1.5:+.2f}) vs 1/T : Eg = {corr['Eg']:.3f} +/- {corr['err']:.3f} eV  (r^2 = {corr['r2']:.5f})")
print(f"\ncompare against the RIGHT reference:")
print(f"  Eg(0)   Varshni, what an Arrhenius slope returns : {EG_0K:.3f} eV")
print(f"  Eg(300) what is usually quoted for germanium     : {varshni(300):.3f} eV")
print(f"\n  deviation of the corrected fit from Eg(0): "
      f"{100*abs(corr['Eg']-EG_0K)/EG_0K:.1f} %")
print(f"  (if you had compared against {varshni(300):.2f} eV instead you would have reported "
      f"{100*abs(corr['Eg']-varshni(300))/varshni(300):.0f} % -- a real result made to look like a bad one)")

intrinsic fit window: 300-350 K
measured mobility exponent from the extrinsic region: alpha = 1.88

  plain  ln(sigma)          vs 1/T : Eg = 0.783 +/- 0.001 eV  (r^2 = 0.99999)
  corr.  ln(sigma T^+0.38) vs 1/T : Eg = 0.804 +/- 0.001 eV  (r^2 = 1.00000)

compare against the RIGHT reference:
  Eg(0)   Varshni, what an Arrhenius slope returns : 0.744 eV
  Eg(300) what is usually quoted for germanium     : 0.664 eV

  deviation of the corrected fit from Eg(0): 8.0 %
  (if you had compared against 0.66 eV instead you would have reported 21 % -- a real result made to look like a bad one)


## 5. Report item 4 — Hall coefficient versus inverse temperature

Three regimes are visible, and the plot is worth reading left to right (high T to low T):

- **Intrinsic (left, small $1/T$).** $n \approx p$ and $R_H \to 0$, because the electron and
  hole contributions nearly cancel. $R_H$ is small and of the sign of the *faster* carrier.
- **Crossover.** $R_H$ passes through zero at $T_0$ and swings hard to the opposite sign.
- **Extrinsic (right, large $1/T$).** $R_H \to 1/(eN_A)$, a plateau set purely by the dopant
  concentration. This is the region that tells you how the crystal was doped.

Plotted on a log $|R_H|$ axis as well, because the magnitudes span two decades and the
plateau is otherwise invisible next to the crossover swing.

In [9]:
c, E = ref, err[REF]
# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

m = np.isfinite(c["R_H"])
ax[0].errorbar(1000 / c["T"][m], c["R_H"][m], yerr=E["sig_RH"][m], fmt=".", ms=4,
               elinewidth=0.6, color="tab:blue")
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$R_H$ (m$^3$/C)")
ax[0].set_title(r"$R_H$ vs inverse temperature", pad=28)
sec = ax[0].secondary_xaxis("top", functions=(lambda x: 1000 / np.where(x == 0, np.nan, x),
                                              lambda t: 1000 / np.where(t == 0, np.nan, t)))
sec.set_xlabel("T (K)")

ax[1].semilogy(1000 / c["T"][m], np.abs(c["R_H"][m]), ".", ms=4)
plateau = np.nanmedian(np.abs(c["R_H"])[c["T"] < 200])
ax[1].axhline(plateau, color="crimson", ls="--", lw=1)
ax[1].annotate(f"extrinsic plateau\n$|R_H|$ = {plateau:.2f} m$^3$/C",
               (1000 / c["T"].min(), plateau), xytext=(-140, -24),
               textcoords="offset points", color="crimson", fontsize=8)
ax[1].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$|R_H|$ (m$^3$/C)",
          title=r"$|R_H|$, log scale — note the sign change dip")

for f, nm in zip(figs, ["item4_RH_vs_invT", "item4_RH_magnitude"]):
    f.suptitle("Hall coefficient", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()


## 6. Report item 6 — $R_H\sigma$ versus T, and where $R_H$ vanishes

$R_H \sigma$ is the **signed Hall mobility**. Multiplying by σ divides out the carrier
density, leaving a quantity with units of mobility that carries the sign of the dominant
carrier — so this plot isolates "which carrier is winning" from "how many carriers there are".

The zero crossing $T_0$ is where

$$p\mu_p^2 = n\mu_n^2$$

This is *not* the extrinsic/intrinsic transition from item 2. It happens considerably later:
intrinsic carriers must not merely appear, but appear in enough numbers that the faster
electrons outweigh the more numerous holes. Expect $T_0$ to sit well above the ρ maximum, and
be ready to explain why they differ.

In [10]:
def find_T0(T, R_H):
    """Temperature where R_H crosses zero: linear interpolation between bracketing points."""
    m = np.isfinite(R_H)
    T, R = T[m], R_H[m]
    i = np.argmax(np.sign(R) != np.sign(R[0]))
    if i == 0:
        return np.nan, None
    T0 = T[i - 1] + (T[i] - T[i - 1]) * (0 - R[i - 1]) / (R[i] - R[i - 1])
    return T0, (T[i - 1], R[i - 1], T[i], R[i])


c = ref
T0, bracket = find_T0(c["T"], c["R_H"])

fig, ax = plt.subplots(figsize=(7.5, 4.6))
m = np.isfinite(c["RH_sigma"])
ax.plot(c["T"][m], c["RH_sigma"][m], ".", ms=5)
ax.axhline(0, color="k", lw=0.8)
ax.axvline(T0, color="crimson", ls="--", lw=1.3)
ax.annotate(f"$T_0$ = {T0:.1f} K\n$R_H$ = 0", (T0, 0), xytext=(14, 40),
            textcoords="offset points", color="crimson",
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1))
ax.axvline(T_rho_max, color="tab:green", ls=":", lw=1.3)
ax.annotate(f"$\\rho$ max\n{T_rho_max:.0f} K", (T_rho_max, 0), xytext=(-70, -48),
            textcoords="offset points", color="tab:green",
            arrowprops=dict(arrowstyle="->", color="tab:green", lw=1))
ax.set(xlabel="T (K)", ylabel=r"$R_H\,\sigma$ (m$^2$/Vs)",
       title=r"Item 6 — $R_H\sigma$ (signed Hall mobility) vs $T$")
plt.tight_layout(); plt.savefig(FIG / "item6_RHsigma_vs_T.png", bbox_inches="tight"); plt.show()

print(f"R_H crosses zero between {bracket[0]:.1f} K ({bracket[1]:+.4f}) "
      f"and {bracket[2]:.1f} K ({bracket[3]:+.4f})")
print(f"   -> T0 = {T0:.1f} K\n")
print(f"compare: rho maximum (extrinsic/intrinsic transition) = {T_rho_max:.0f} K")
print(f"         difference = {T0 - T_rho_max:.0f} K")
print("\nThe two are different events. At the rho maximum, generated carriers start to")
print("dominate CONDUCTION. At T0, they have grown enough that the faster electrons exactly")
print("cancel the more numerous holes in the HALL voltage, which takes another ~30 K.")

print("\nT0 across all conditions:")
for name, cc in conditions.items():
    if cc["has_hall"]:
        t0, _ = find_T0(cc["T"], cc["R_H"])
        print(f"   {name:<14s} ({cc['I_uA']:>3.0f} uA, {cc['B_T']/G2T:>4.0f} G):  T0 = {t0:.1f} K")

R_H crosses zero between 271.2 K (+0.0036) and 276.0 K (-0.1211)
   -> T0 = 271.3 K

compare: rho maximum (extrinsic/intrinsic transition) = 242 K
         difference = 29 K

The two are different events. At the rho maximum, generated carriers start to
dominate CONDUCTION. At T0, they have grown enough that the faster electrons exactly
cancel the more numerous holes in the HALL voltage, which takes another ~30 K.

T0 across all conditions:
   neg_1500_100   (100 uA, 1777 G):  T0 = 275.1 K
   3000_100       (100 uA, 3490 G):  T0 = 271.3 K
   3000_500       ( 10 uA, 3618 G):  T0 = 271.2 K
   neg_1500_500   (500 uA, 1784 G):  T0 = 277.7 K


## 7. Report items 5, 10, 11 — carrier concentration and mobilities

### Item 10 — carrier concentration in the extrinsic region

In the extrinsic plateau one carrier type dominates and the Hall coefficient reduces to

$$R_H = \frac{r_H}{N_A e} \;\Longrightarrow\; N_A = \frac{r_H}{e\,|R_H^\text{plateau}|}$$

$r_H$ is the **Hall scattering factor**, which accounts for the fact that carriers have a
distribution of velocities rather than all moving at the drift velocity. For lattice
scattering $r_H = 3\pi/8 \approx 1.18$; for ionised-impurity scattering it is about 1.93.
Most lab treatments set $r_H = 1$ and thereby carry an unremoved ~18 % systematic. Both are
reported below.

### Item 11 — the two mobilities in the extrinsic region

The measured Hall mobility in the plateau is the **hole** mobility, since holes are the only
carriers present:

$$\mu_p = |R_H|\,\sigma$$

The electron mobility there cannot be measured directly — there are essentially no electrons
in the extrinsic region. It is obtained from the mobility ratio $b$ found at $T_0$:
$\mu_n = b\,\mu_p$.

### Item 5 — the mobility ratio $b = \mu_n/\mu_p$, two independent ways

**Method A — conductivity extrapolation (Melissinos §3.3).** At $T_0$, $R_H = 0$ gives
$p\mu_p^2 = n\mu_n^2$, so $n/p = 1/b^2$. Then:

- measured there: $\sigma(T_0) = e\mu_p(p + nb) = e\mu_p p(1 + 1/b)$
- dopant holes alone: $\sigma_\text{ext}(T_0) = e\mu_p N_A = e\mu_p(p-n) = e\mu_p p(1 - 1/b^2)$

Dividing, everything unknown cancels:

$$r = \frac{\sigma(T_0)}{\sigma_\text{ext}(T_0)} = \frac{1}{1 - 1/b}
\;\Longrightarrow\; \boxed{\,b = \frac{r}{r-1}\,}$$

$\sigma_\text{ext}(T_0)$ is hypothetical — the conductivity the sample *would* have at $T_0$
if only the dopant holes existed — so it must be constructed by fitting the power law
$\sigma \propto T^{-\alpha}$ in the extrinsic region (where $p = N_A$ is frozen and only
mobility varies) and continuing it up to $T_0$. The assumption is that $\mu_p(T)$ keeps
following lattice scattering above the transition, which it does: phonon scattering does not
care whether electrons have appeared alongside.

**Method B — the shape of the $R_H$ curve.** Writing the two-carrier Hall coefficient with
$b = \mu_n/\mu_p$ and charge neutrality $p - n = N_A$:

$$R_H = \frac{p - nb^2}{e\,(p+nb)^2}$$

Setting $dR_H/dn = 0$ gives the extremum at $n = N_A/(b-1)$, and substituting back:

$$\frac{|R_H^\text{extremum}|}{|R_H^\text{plateau}|} = \frac{(b-1)^2}{4b}$$

One quadratic in $b$, using only two values read off the $R_H$ curve — no fit window, no
extrapolation, no conductivity data. If A and B agree, the answer is solid.

In [11]:
EXTRINSIC = (110.0, 200.0)      # window that is cleanly extrinsic, from item 2
r_H_lattice = 3 * np.pi / 8     # Hall scattering factor for lattice scattering


def mobility_analysis(name, extrinsic=EXTRINSIC, verbose=True):
    c = conditions[name]
    T, sigma, R_H = c["T"], c["sigma"], c["R_H"]
    T0, _ = find_T0(T, R_H)

    # --- extrinsic plateau: carrier concentration and hole mobility (items 10, 11) ---
    m = (T >= extrinsic[0]) & (T <= extrinsic[1]) & np.isfinite(R_H)
    RH_plateau = np.nanmedian(np.abs(R_H[m]))
    N_A_raw = 1 / (e * RH_plateau)
    N_A_corr = r_H_lattice / (e * RH_plateau)
    mu_p_ext = RH_plateau * np.nanmedian(sigma[m])

    # --- method A: conductivity extrapolation ---
    fitm = m & np.isfinite(sigma) & (sigma > 0)
    alpha, icept = np.polyfit(np.log(T[fitm]), np.log(sigma[fitm]), 1)
    sigma_ext = lambda t: np.exp(icept + alpha * np.log(np.asarray(t, float)))
    sig_meas_T0 = at_T(c, "sigma", T0)
    sig_ext_T0 = float(sigma_ext(T0))
    r_ratio = sig_meas_T0 / sig_ext_T0
    b_A = r_ratio / (r_ratio - 1) if r_ratio > 1 else np.nan

    # --- method B: shape of R_H ---
    # |R_H^extremum| / |R_H^plateau| = (b-1)^2 / 4b, solved as a quadratic in b.
    # The "plateau" is the saturation value 1/(e N_A) -- the same median over the
    # extrinsic window used for item 10, so the two answers stay consistent. Using the
    # peak |R_H| instead shifts b slightly; both are reported.
    fin = np.isfinite(R_H)
    s0 = np.sign(R_H[fin][0])
    extm = np.abs(R_H[fin][np.sign(R_H[fin]) == -s0]).max()
    peak = np.abs(R_H[fin][np.sign(R_H[fin]) == s0]).max()

    def solve_b(ratio):
        q = 2 + 4 * ratio
        return (q + np.sqrt(q ** 2 - 4)) / 2

    b_B = solve_b(extm / RH_plateau)
    b_B_peak = solve_b(extm / peak)

    mu_p_T0 = sig_ext_T0 / (e * N_A_raw)
    res = dict(run=name, T0=T0, alpha=alpha, r=r_ratio, b_A=b_A, b_B=b_B,
               b_B_peak=b_B_peak, RH_extremum=extm, RH_peak=peak,
               N_A=N_A_raw, N_A_corr=N_A_corr, RH_plateau=RH_plateau,
               mu_p_ext=mu_p_ext, mu_n_ext=b_A * mu_p_ext,
               mu_p_T0=mu_p_T0, mu_n_T0=b_A * mu_p_T0,
               sigma_ext=sigma_ext, sig_meas_T0=sig_meas_T0, sig_ext_T0=sig_ext_T0)
    return res


M = {n: mobility_analysis(n) for n, c in conditions.items() if c["has_hall"]}
mob = M[REF]

print(f"=== {REF}  (extrinsic window {EXTRINSIC[0]:.0f}-{EXTRINSIC[1]:.0f} K) ===\n")
print("Item 10 - carrier concentration in the extrinsic region")
print(f"   |R_H| plateau           = {mob['RH_plateau']:.3f} m^3/C")
print(f"   N_A  (r_H = 1)          = {mob['N_A']:.3e} m^-3 = {mob['N_A']*1e-6:.3e} cm^-3")
print(f"   N_A  (r_H = 3pi/8=1.18) = {mob['N_A_corr']:.3e} m^-3 = {mob['N_A_corr']*1e-6:.3e} cm^-3")

print("\nItem 11 - mobilities in the extrinsic region")
print(f"   sigma ~ T^{mob['alpha']:.2f}   (pure acoustic-phonon scattering gives -1.5;")
print(f"                    more negative indicates ionised-impurity scattering too)")
print(f"   mu_p (plateau mean)     = {mob['mu_p_ext']:.3f} m^2/Vs = {mob['mu_p_ext']*1e4:.0f} cm^2/Vs")
print(f"   mu_n = b*mu_p           = {mob['mu_n_ext']:.3f} m^2/Vs = {mob['mu_n_ext']*1e4:.0f} cm^2/Vs")

print("\nItem 5 - mobility ratio b = mu_n/mu_p")
print(f"   T0                      = {mob['T0']:.1f} K")
print(f"   sigma measured at T0    = {mob['sig_meas_T0']:.4f} S/m")
print(f"   sigma extrapolated      = {mob['sig_ext_T0']:.4f} S/m")
print(f"   r                       = {mob['r']:.3f}")
print(f"   METHOD A  b = r/(r-1)   = {mob['b_A']:.2f}")
print(f"   METHOD B  from R_H shape= {mob['b_B']:.2f}   "
      f"(using the peak instead of the plateau: {mob['b_B_peak']:.2f})")
print(f"   germanium literature    : mu_n/mu_p = 2.1")
print(f"\n   Methods A and B share no assumptions -- A needs a power-law fit and an")
print(f"   extrapolation, B needs only two values read off the R_H curve. Agreement to")
print(f"   {100*abs(mob['b_A']-mob['b_B'])/mob['b_A']:.0f} % is the strongest evidence in this report that b is real.")

=== 3000_100  (extrinsic window 110-200 K) ===

Item 10 - carrier concentration in the extrinsic region
   |R_H| plateau           = 2.914 m^3/C
   N_A  (r_H = 1)          = 2.142e+18 m^-3 = 2.142e+12 cm^-3
   N_A  (r_H = 3pi/8=1.18) = 2.524e+18 m^-3 = 2.524e+12 cm^-3

Item 11 - mobilities in the extrinsic region
   sigma ~ T^-1.88   (pure acoustic-phonon scattering gives -1.5;
                    more negative indicates ionised-impurity scattering too)
   mu_p (plateau mean)     = 0.795 m^2/Vs = 7946 cm^2/Vs
   mu_n = b*mu_p           = 1.229 m^2/Vs = 12292 cm^2/Vs

Item 5 - mobility ratio b = mu_n/mu_p
   T0                      = 271.3 K
   sigma measured at T0    = 0.2694 S/m
   sigma extrapolated      = 0.0952 S/m
   r                       = 2.828
   METHOD A  b = r/(r-1)   = 1.55
   METHOD B  from R_H shape= 1.58   (using the peak instead of the plateau: 1.54)
   germanium literature    : mu_n/mu_p = 2.1

   Methods A and B share no assumptions -- A needs a power-law fit and an


In [12]:
c, mob = ref, M[REF]
# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

# --- left: the extrapolation that gives b ---
m = np.isfinite(c["sigma"])
ax[0].semilogy(c["T"][m], c["sigma"][m], ".", ms=4, color="0.6", label="measured $\\sigma$")
tt = np.linspace(EXTRINSIC[0], mob["T0"] * 1.06, 200)
ax[0].semilogy(tt, mob["sigma_ext"](tt), "--", color="crimson", lw=1.6,
               label=r"extrinsic power law $\sigma\propto T^{%.2f}$" % mob["alpha"])
ax[0].axvspan(*EXTRINSIC, color="tab:blue", alpha=0.12, label="extrinsic fit window")
ax[0].axvline(mob["T0"], color="tab:green", ls=":", lw=1.5)
ax[0].plot([mob["T0"]] * 2, [mob["sig_ext_T0"], mob["sig_meas_T0"]], "o-", color="k",
           lw=1.5, ms=5)
ax[0].annotate(f"$r$ = {mob['r']:.2f}\n$\\Rightarrow b$ = {mob['b_A']:.2f}",
               (mob["T0"], np.sqrt(mob["sig_ext_T0"] * mob["sig_meas_T0"])),
               xytext=(14, 6), textcoords="offset points", fontsize=10,
               bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", alpha=0.9))
ax[0].annotate(f"$T_0$ = {mob['T0']:.0f} K", (mob["T0"], np.nanmin(c["sigma"]) * 1.15),
               xytext=(-62, 0), textcoords="offset points", color="tab:green", fontsize=9)
ax[0].set(xlabel="T (K)", ylabel=r"$\sigma$ (S/m)", title="Method A: extrapolate $\\sigma$ to $T_0$")
ax[0].legend(fontsize=8, loc="upper left")

# --- right: the shape method ---
fin = np.isfinite(c["R_H"])
s0 = np.sign(c["R_H"][fin][0])
plat, extm = mob["RH_plateau"], mob["RH_extremum"]
ax[1].plot(c["T"][fin], c["R_H"][fin], ".", ms=4)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].axhline(s0 * plat, color="tab:blue", ls="--", lw=1.2)        # plateau = 1/(e N_A)
ax[1].axhline(-s0 * extm, color="tab:orange", ls="--", lw=1.2)     # extremum, past T0
ax[1].axvspan(*EXTRINSIC, color="tab:blue", alpha=0.10)
ax[1].text(0.04, 0.22, f"plateau $|R_H|$ = {plat:.2f}\n(= $1/eN_A$)",
           transform=ax[1].transAxes, color="tab:blue", fontsize=8.5,
           bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.85))
ax[1].annotate(f"extremum $|R_H|$ = {extm:.3f}", xy=(0.88, 0.955), xytext=(0.47, 0.72),
               xycoords="axes fraction", textcoords="axes fraction", color="tab:orange",
               fontsize=8.5, arrowprops=dict(arrowstyle="->", color="tab:orange", lw=1))
ax[1].set(xlabel="T (K)", ylabel=r"$R_H$ (m$^3$/C)")
ax[1].set_title(f"Method B: $(b-1)^2/4b$ = {extm/plat:.4f} $\\to$ $b$ = {mob['b_B']:.2f}")

for f, nm in zip(figs, ["item5_mobility_ratio_methodA", "item5_mobility_ratio_methodB"]):
    f.suptitle("Item 5 — mobility ratio, two independent methods", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

# sensitivity of b to the extrinsic window -- this, not instrument noise, is the error on b
print("sensitivity of method A to the extrinsic fit window:")
for w in [(100, 180), (110, 200), (120, 210), (130, 220), (150, 230)]:
    mm = mobility_analysis(REF, extrinsic=w)
    print(f"   {w[0]}-{w[1]} K:  sigma ~ T^{mm['alpha']:.2f},  r = {mm['r']:.2f},  b = {mm['b_A']:.2f}")
print("\nThe spread across windows IS the uncertainty on b -- it dominates anything the")
print("electronics contribute.")

sensitivity of method A to the extrinsic fit window:
   100-180 K:  sigma ~ T^-1.82,  r = 2.73,  b = 1.58
   110-200 K:  sigma ~ T^-1.88,  r = 2.83,  b = 1.55
   120-210 K:  sigma ~ T^-1.91,  r = 2.87,  b = 1.53
   130-220 K:  sigma ~ T^-1.94,  r = 2.92,  b = 1.52
   150-230 K:  sigma ~ T^-1.97,  r = 2.95,  b = 1.51

The spread across windows IS the uncertainty on b -- it dominates anything the
electronics contribute.


## 8. Report item 7 — p-type or n-type, and the field-direction question

### The physics

Holes and electrons drift in *opposite* directions under the same applied field, and carry
*opposite* charge. The Lorentz force $q(\vec v\times\vec B)$ therefore pushes both toward the
same transverse face. What differs is the sign of the charge that accumulates there, so:

$$R_H = +\frac{1}{pe}\ \text{(holes)}, \qquad R_H = -\frac{1}{ne}\ \text{(electrons)}$$

### Why you cannot simply read the sign off the plot

Two conventions of the apparatus, neither of them physics, each flip the sign of every
measured $R_H$: which lead the electrometer calls "+", and which direction the gaussmeter
calls positive field. An absolute sign taken from the raw data is therefore not trustworthy
on its own.

### The argument that does work

The two ends of the $R_H$ curve have a **fixed relationship** that no wiring convention can
change:

1. **The extrinsic plateau carries the sign of the dopant**, because those are the only
   carriers present.
2. **The intrinsic end carries the sign of the faster carrier.** When $n \approx p$,
   $R_H \propto p\mu_p^2 - n\mu_n^2$, which is dominated by whichever mobility is larger. In
   germanium that is always the electron ($\mu_n > \mu_p$), regardless of doping.

So the high-temperature end is a **sign reference**: whatever sign your data shows there means
"electron-like" in your convention. If the plateau has the opposite sign, the dopant is an
acceptor and the sample is p-type.

This is precisely the manual's question, *"why don't you need to know the direction of the
magnetic field after all?"* — the physical content is the relationship between the two ends,
not the sign printed on either one.

### An independent confirmation

The mobility ratio from §7 is a second, independent check. Method A returns the
*minority-to-majority* mobility ratio at $T_0$ whichever way the sample is doped (run the
same algebra with $n$ and $p$ swapped and $r/(r-1)$ comes out as $\mu_p/\mu_n$). A value
above 1 therefore says the carriers arriving at $T_0$ are faster than the ones already there
— i.e. electrons arriving into a hole-dominated sample.

### What this means for every other plot

Once the argument above has been made, the sign is no longer unknown, so there is no reason
to keep plotting the apparatus's inverted version. `RH_SIGN = -1` is applied in §1.4, and
every other $R_H$ plot in this notebook shows the **physical** sign:

$$R_H > 0 \\ \text{in the extrinsic plateau (holes)} \quad\longrightarrow\quad
R_H < 0 \\ \text{above } T_0 \\ \text{(electrons)}$$

Only the right-hand panel in this section is left uncorrected, because it is the evidence.
No magnitude, and no derived quantity ($N_A$, $b$, $T_0$, $\mu$), changes — all of those
depend on $|R_H|$ or on the *relative* sign of the two ends, both of which are invariant
under a global flip.


In [13]:
# This section argues from the UNCORRECTED values. The whole point of item 7 is that the
# absolute sign the apparatus reports cannot be trusted, so the argument has to be made
# before RH_SIGN is invoked. R_H_raw is what the electrometer and gaussmeter gave.
c = ref
RH_raw = c["R_H_raw"]
fin = np.isfinite(RH_raw)
plateau_signed = np.nanmedian(RH_raw[fin & (c["T"] < 200)])
highT_signed = np.nanmedian(RH_raw[fin & (c["T"] > 320)])

# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

# left: raw diagonal signal at each field state -- shows the Hall part flipping with B
st = runs[REF]
for lab, col in (("+", "tab:red"), ("0", "0.5"), ("-", "tab:blue")):
    if lab in st:
        ax[0].plot(st[lab]["T"], st[lab]["R_diag"], ".", ms=4, color=col,
                   label=f"{lab}B  ({np.nanmedian(st[lab]['B_G']):+.0f} G)")
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set(xlabel="T (K)", ylabel=r"$R_\mathrm{diag}$ ($\Omega$)",
          title="Hall signal reverses with the field, offset does not")
ax[0].legend(fontsize=8)

# right: R_H with both ends marked
ax[1].plot(c["T"][fin], RH_raw[fin], ".", ms=4, color="k")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].fill_between(c["T"][fin], 0, np.where(RH_raw[fin] > 0, RH_raw[fin], 0),
                   color="tab:orange", alpha=0.25)
ax[1].fill_between(c["T"][fin], 0, np.where(RH_raw[fin] < 0, RH_raw[fin], 0),
                   color="tab:blue", alpha=0.25)
ax[1].annotate(f"extrinsic plateau: {plateau_signed:+.2f}\n= sign of the DOPANT",
               (150, plateau_signed), xytext=(10, 40), textcoords="offset points",
               fontsize=9, arrowprops=dict(arrowstyle="->", lw=1))
ax[1].annotate(f"intrinsic end: {highT_signed:+.3f}\n= sign of the FASTER carrier (electron)",
               (335, highT_signed), xytext=(-245, -55), textcoords="offset points",
               fontsize=9, arrowprops=dict(arrowstyle="->", lw=1))
ax[1].set(xlabel="T (K)", ylabel=r"$R_H$ as measured (m$^3$/C)",
          title="The two ends have opposite signs (uncorrected sign)")

for f, nm in zip(figs, ["item7_field_reversal", "item7_carrier_type"]):
    f.suptitle("Item 7 — determining the carrier type", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

print(f"measured extrinsic plateau R_H : {plateau_signed:+.3f} m^3/C")
print(f"measured intrinsic end     R_H : {highT_signed:+.3f} m^3/C")
print(f"mobility ratio from section 7  : b = {M[REF]['b_A']:.2f}  (> 1)\n")
print("Chain of reasoning:")
print(f"  1. The intrinsic end is {'positive' if highT_signed > 0 else 'negative'}. In Ge the intrinsic end must carry the")
print(f"     electron sign, so in THIS apparatus's convention, "
      f"{'positive' if highT_signed > 0 else 'negative'} = electron-like.")
print(f"  2. The extrinsic plateau is {'positive' if plateau_signed > 0 else 'negative'}, i.e. the opposite sign,")
print(f"     so the dopant carriers are holes.")
print(f"  3. b = {M[REF]['b_A']:.2f} > 1 independently confirms that the carriers arriving at T0 are")
print(f"     faster than those already present -- electrons arriving into a hole-doped sample.")
print(f"\n  ==> the sample is P-TYPE (acceptor-doped germanium)")
print(f"\nNote the crossing itself is what proves this. An n-type sample would have NO")
print("sign change at all: electrons dominate the plateau AND electrons are still the")
print("faster carrier intrinsically, so both ends would carry the same sign. The measured")
print(f"plateau is {plateau_signed:+.2f}; the textbook value for holes is +1/(N_A e) > 0, so the")
print(f"apparatus convention is inverted (RH_SIGN = {RH_SIGN}, applied in section 1.4). Every")
print("OTHER R_H plot in this notebook therefore carries the PHYSICAL sign: positive in")
print("the extrinsic plateau, crossing to negative above T0. Only the panel above is")
print("uncorrected. No magnitude anywhere in the report changes -- only the sign.")

measured extrinsic plateau R_H : -2.881 m^3/C
measured intrinsic end     R_H : +0.020 m^3/C
mobility ratio from section 7  : b = 1.55  (> 1)

Chain of reasoning:
  1. The intrinsic end is positive. In Ge the intrinsic end must carry the
     electron sign, so in THIS apparatus's convention, positive = electron-like.
  2. The extrinsic plateau is negative, i.e. the opposite sign,
     so the dopant carriers are holes.
  3. b = 1.55 > 1 independently confirms that the carriers arriving at T0 are
     faster than those already present -- electrons arriving into a hole-doped sample.

  ==> the sample is P-TYPE (acceptor-doped germanium)

Note the crossing itself is what proves this. An n-type sample would have NO
sign change at all: electrons dominate the plateau AND electrons are still the
faster carrier intrinsically, so both ends would carry the same sign. The measured
plateau is -2.88; the textbook value for holes is +1/(N_A e) > 0, so the
apparatus convention is inverted (RH_SIGN = -1

## 9. Report items 8, 9 — carrier concentration and Hall mobility versus temperature

### Item 8 — carrier concentration

From $R_H = r_H/(pe)$, inverting gives the concentration at every temperature:

$$p(T) = \frac{r_H}{e\,|R_H(T)|}$$

**This formula is only valid where one carrier type dominates.** In the mixed region both
species contribute to $R_H$, they partly cancel, $|R_H|$ collapses toward zero, and
$1/(e|R_H|)$ therefore *diverges*. The apparent spike near $T_0$ in the plot is not a real
carrier population — it is the formula breaking down. The plot shades that region out.

At high temperature, past $T_0$, a genuine intrinsic concentration can be recovered from the
activation fit rather than from $R_H$ directly, since there $n = p = n_i \propto
T^{3/2}e^{-E_g/2k_BT}$.

### Item 9 — Hall mobility

$$\mu_H = |R_H|\,\sigma$$

In the extrinsic region this is the true hole mobility (times $r_H$). Its temperature
dependence is the physics of scattering: a $T^{-3/2}$ power law indicates acoustic-phonon
(lattice) scattering, while a shallower or rising trend at low T indicates ionised-impurity
scattering, which weakens as carriers speed up.

In [14]:
c, mob = ref, M[REF]
fin = np.isfinite(c["R_H"]) & np.isfinite(c["carrier"])
valid = fin & (np.abs(c["R_H"]) > 0.3 * mob["RH_plateau"])     # single-carrier region only

# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

# --- carrier concentration ---
ax[0].semilogy(1000 / c["T"][fin], c["carrier"][fin] * 1e-6, ".", ms=4, color="0.7",
               label="all data")
ax[0].semilogy(1000 / c["T"][valid], c["carrier"][valid] * 1e-6, ".", ms=5, color="tab:blue",
               label="single-carrier region (valid)")
ax[0].axhline(mob["N_A"] * 1e-6, color="crimson", ls="--", lw=1)
ax[0].annotate(f"$N_A$ = {mob['N_A']*1e-6:.2e} cm$^{{-3}}$",
               (1000 / c["T"].min(), mob["N_A"] * 1e-6), xytext=(-165, 12),
               textcoords="offset points", color="crimson", fontsize=8)
bad_lo, bad_hi = 1000 / (mob["T0"] + 40), 1000 / (mob["T0"] - 40)
ax[0].axvspan(bad_lo, bad_hi, color="tab:red", alpha=0.10)
ax[0].annotate("mixed carriers:\n$1/e|R_H|$ diverges", ((bad_lo + bad_hi) / 2, 3e13),
               ha="center", fontsize=8, color="tab:red")
ax[0].set(xlabel="1000/T (K$^{-1}$)", ylabel=r"$1/(e|R_H|)$  (cm$^{-3}$)")
ax[0].set_title("Item 8 — apparent carrier concentration", pad=28)
ax[0].legend(fontsize=8, loc="lower left")
sec = ax[0].secondary_xaxis("top", functions=(lambda x: 1000 / np.where(x == 0, np.nan, x),
                                              lambda t: 1000 / np.where(t == 0, np.nan, t)))
sec.set_xlabel("T (K)")

# --- Hall mobility ---
ax[1].loglog(c["T"][fin], c["mu_hall"][fin] * 1e4, ".", ms=4, color="0.7", label="all data")
ax[1].loglog(c["T"][valid], c["mu_hall"][valid] * 1e4, ".", ms=5, color="tab:blue",
             label="single-carrier region")
m_ext = (c["T"] >= EXTRINSIC[0]) & (c["T"] <= EXTRINSIC[1]) & fin
sl, ic = np.polyfit(np.log(c["T"][m_ext]), np.log(c["mu_hall"][m_ext]), 1)
tt = np.linspace(EXTRINSIC[0] * 0.9, EXTRINSIC[1] * 1.15, 50)
ax[1].loglog(tt, np.exp(ic + sl * np.log(tt)) * 1e4, "--", color="crimson", lw=1.5,
             label=fr"$\mu_H \propto T^{{{sl:.2f}}}$")
ax[1].set(xlabel="T (K)", ylabel=r"$\mu_H = |R_H|\sigma$  (cm$^2$/Vs)")
ax[1].set_title("Item 9 — Hall mobility", pad=28)
# clip the y-range to the physically meaningful points: right at T0, |R_H| -> 0 drags
# mu_H = |R_H|sigma to nearly zero, which is the formula failing, not a real mobility
lo = np.nanmin(c["mu_hall"][valid]) * 1e4
ax[1].set_ylim(lo * 0.45, np.nanmax(c["mu_hall"][valid]) * 1e4 * 2.2)
ax[1].legend(fontsize=8, loc="lower left")

for f, nm in zip(figs, ["item8_carrier_concentration", "item9_hall_mobility"]):
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

print(f"extrinsic-region Hall mobility scales as T^{sl:.2f}")
print(f"   (-1.5 = pure acoustic-phonon scattering; Ge holes are often steeper, ~T^-2.3)")
print(f"mu_H at 150 K = {at_T(c,'mu_hall',150)*1e4:.0f} cm^2/Vs")
print(f"mu_H at 300 K = {at_T(c,'mu_hall',300)*1e4:.0f} cm^2/Vs")
print(f"\ncarrier concentration in the extrinsic plateau = {mob['N_A']*1e-6:.3e} cm^-3")
print(f"at 300 K the formula 1/(e|R_H|) gives {at_T(c,'carrier',300)*1e-6:.3e} cm^-3, which is")
print(f"NOT a carrier density: 300 K is above T0 = {mob['T0']:.0f} K, inside the mixed region.")

extrinsic-region Hall mobility scales as T^-1.62
   (-1.5 = pure acoustic-phonon scattering; Ge holes are often steeper, ~T^-2.3)
mu_H at 150 K = 8411 cm^2/Vs
mu_H at 300 K = 1209 cm^2/Vs

carrier concentration in the extrinsic plateau = 2.142e+12 cm^-3
at 300 K the formula 1/(e|R_H|) gives 6.941e+13 cm^-3, which is
NOT a carrier density: 300 K is above T0 = 271 K, inside the mixed region.


## 10. Report item 12 — magnetoresistance

Compare ρ measured with the field on against ρ measured at zero field. Because the multifield
program takes a ~0 G point between every ±B pair, each file supplies its **own** zero-field
reference at the same current and essentially the same temperature — a far cleaner comparison
than using a separate run.

### Why resistivity changes at all

In a simple one-carrier Drude picture with a single relaxation time there is *no*
magnetoresistance: the Hall field exactly cancels the Lorentz deflection. Any observed effect
therefore comes from the assumption failing. Two mechanisms matter here:

- **Distribution of velocities.** Carriers do not all move at the drift velocity, so the Hall
  field can only cancel the Lorentz force *on average*. Faster and slower carriers are
  over- and under-deflected, lengthening their paths.
- **Two carrier types.** Near the intrinsic region both electrons and holes are present and
  deflect in the same transverse direction, so the Hall field cannot cancel both.

Both give $\Delta\rho/\rho_0 \propto (\mu B)^2$ at low field, so the effect should be largest
where the mobility is largest — at low temperature.

In [15]:
# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(2)]
ax = [f.add_subplot(111) for f in figs]

mr_rows = []
for name, c in conditions.items():
    if not c["has_hall"] or c["rho_zero"] is None:
        continue
    frac = c["rho"] / c["rho_zero"] - 1
    lbl = f"{c['B_T']/G2T:.0f} G, {c['I_uA']:.0f} $\\mu$A"
    ax[0].plot(c["T"], 100 * frac, lw=1.5, label=lbl)
    for Tq in (110, 150, 200, 250, 300):
        mr_rows.append(dict(run=name, B_G=round(c["B_T"] / G2T), I_uA=c["I_uA"], T=Tq,
                            pct=100 * float(np.interp(Tq, c["T"], frac)),
                            mu_B=at_T(c, "mu_hall", Tq) * c["B_T"]))
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set(xlabel="T (K)", ylabel=r"$\Delta\rho/\rho_0$ (%)",
          title="Magnetoresistance vs temperature")
ax[0].legend(fontsize=8)

mr = pd.DataFrame(mr_rows)
for Tq, mk in ((110, "o"), (150, "D"), (200, "s"), (250, "v"), (300, "^")):
    sel = mr[mr["T"] == Tq]
    ax[1].plot(sel["mu_B"] ** 2, sel["pct"], mk, ms=7, label=f"{Tq} K")

# low-field quadratic law, fitted through the origin using only the small-(muB)^2 points
low = mr[mr["mu_B"] ** 2 < 0.05]
slope_lf = np.sum(low["mu_B"] ** 2 * low["pct"]) / np.sum(low["mu_B"] ** 4)
xx = np.linspace(0, mr["mu_B"].max() ** 2 * 1.05, 50)
ax[1].plot(xx, slope_lf * xx, "--", color="0.4", lw=1.3,
           label=r"low-field fit $\propto(\mu_H B)^2$")
ax[1].set(xlabel=r"$(\mu_H B)^2$", ylabel=r"$\Delta\rho/\rho_0$ (%)",
          title=r"scaling against $(\mu_H B)^2$",
          ylim=(-2, mr["pct"].max() * 1.25))
ax[1].annotate("low-field law fails\nonce $\\mu_H B \\sim 0.5$", (0.23, 38),
               xytext=(-30, 26), textcoords="offset points", fontsize=8.5, color="0.35",
               arrowprops=dict(arrowstyle="->", color="0.45", lw=1))
ax[1].legend(fontsize=8, loc="upper left")

for f, nm in zip(figs, ["item12_magnetoresistance_vs_T", "item12_magnetoresistance_scaling"]):
    f.suptitle("Item 12 — magnetoresistance", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

print("Delta rho / rho_0  (%):\n")
print(mr.pivot_table(index="T", columns=["B_G", "I_uA"], values="pct").round(1).to_string())
print("\nThe effect is largest at low temperature, where the mobility is largest, and grows")
print("with field -- both consistent with a (mu_H B)^2 dependence.")
print("\nThe right panel tests that scaling directly. The small-(mu B)^2 points do fall on")
print("a straight line through the origin, confirming the quadratic law. The points at")
print("110 K and high field sit BELOW that line: the quadratic law is the leading term of")
print("a low-field expansion, and by then mu_H B has reached ~0.5 (see the table in item")
print("13), so the expansion is no longer valid and the magnetoresistance begins to")
print("saturate. That departure is a result, not a discrepancy -- it marks the boundary")
print("of the regime in which the simple Hall analysis of this whole report applies.")

Delta rho / rho_0  (%):

B_G   1777  1784  3490  3618
I_uA   100   500   100   10 
T                           
110   23.3  23.5  38.6  37.5
150   12.8  12.8  25.5  25.1
200    5.9   6.0  14.6  14.5
250    3.1   3.1   8.8   8.7
300    0.7   0.7   3.6   3.6

The effect is largest at low temperature, where the mobility is largest, and grows
with field -- both consistent with a (mu_H B)^2 dependence.

The right panel tests that scaling directly. The small-(mu B)^2 points do fall on
a straight line through the origin, confirming the quadratic law. The points at
110 K and high field sit BELOW that line: the quadratic law is the leading term of
a low-field expansion, and by then mu_H B has reached ~0.5 (see the table in item
13), so the expansion is no longer valid and the magnetoresistance begins to
saturate. That departure is a result, not a discrepancy -- it marks the boundary
of the regime in which the simple Hall analysis of this whole report applies.


## 11. Report item 13 — comparing the different experiments

The manual requires runs at several currents and fields precisely so that you can check
**which results are properties of the sample and which are artefacts of the measurement**.
The material properties — ρ, $R_H$, $T_0$, $N_A$, $b$ — must not depend on either knob. Any
spread is a systematic uncertainty that belongs in the report.

Your four runs vary both knobs:

| run | field | current | what it isolates |
|---|---|---|---|
| `neg_1500_100` | 1750 G | 100 µA | lower field, same current as `3000_100` |
| `3000_100` | 3490 G | 100 µA | reference |
| `3000_500` | 3618 G | 10 µA | ~same field, 10× lower current |
| `0_100` | 0 G | 10 µA | zero-field resistivity only |

Comparing `neg_1500_100` with `3000_100` isolates the **field** dependence at fixed current.
Comparing `3000_100` with `3000_500` isolates the **current** dependence at nearly fixed
field. Two effects are expected to be real rather than error:

- **ρ depends on field** through the magnetoresistance of item 12, so ρ from a 3490 G run
  should legitimately exceed ρ from a 1750 G run.
- **Low current degrades precision.** The manual warns that below ~100 µA the voltages become
  too small to read accurately. The 10 µA runs should be visibly noisier.

In [16]:
rows = []
for name, c in conditions.items():
    row = dict(run=name, B_G=round(c["B_T"] / G2T) if c["has_hall"] else 0, I_uA=c["I_uA"],
               rho_300=at_T(c, "rho"), rho_150=at_T(c, "rho", 150),
               noise_rho=scatter(c["rho"]) / np.nanmedian(c["rho"]) * 100)
    if c["has_hall"]:
        mm = M[name]
        row.update(R_H_300=at_T(c, "R_H"), T0=mm["T0"], N_A_cm3=mm["N_A"] * 1e-6,
                   b=mm["b_A"], mu_H_300=at_T(c, "mu_hall") * 1e4,
                   noise_RH=scatter(c["R_H"]) / mm["RH_plateau"] * 100)
    rows.append(row)
comp = pd.DataFrame(rows).sort_values(["I_uA", "B_G"])
print("Comparison across runs\n")
print(comp.to_string(index=False, float_format=lambda v: f"{v:,.4g}"))

print("\nspread of each material property across the three Hall runs:")
h = comp.dropna(subset=["T0"])
for col, unit in (("T0", "K"), ("N_A_cm3", "cm^-3"), ("b", ""), ("mu_H_300", "cm^2/Vs"),
                  ("R_H_300", "m^3/C"), ("rho_150", "Ohm m")):
    v = h[col].to_numpy(float)
    print(f"   {col:<10s} mean {np.mean(v):>11.4g} {unit:<9s} spread "
          f"{np.std(v, ddof=1)/abs(np.mean(v))*100:5.1f} %")

print("\nmeasurement quality (point-to-point scatter, % of typical value):")
for _, r in comp.iterrows():
    nr = f"{r['noise_RH']:.2f}" if np.isfinite(r.get("noise_RH", np.nan)) else "  - "
    print(f"   {r['run']:<14s} {r['I_uA']:>3.0f} uA:  rho {r['noise_rho']:.2f} %,  R_H {nr} %")

Comparison across runs

         run  B_G  I_uA  rho_300  rho_150  noise_rho  R_H_300    T0   N_A_cm3     b  mu_H_300  noise_RH
       0_100    0    10   0.7137    2.704     0.2167      NaN   NaN       NaN   NaN       NaN       NaN
    3000_500 3618    10   0.7458    3.506     0.2527 -0.08716 271.2  2.22e+12 1.577     1,170   0.04674
neg_1500_100 1777   100   0.7334    3.048     0.2209 -0.07428 275.1 2.009e+12 1.391     1,016   0.04715
    3000_100 3490   100   0.7459    3.434     0.2362 -0.09011 271.3 2.142e+12 1.547     1,209   0.04002
neg_1500_500 1784   500   0.7344    3.042     0.2424 -0.07463 277.7 2.018e+12  1.34     1,020   0.04584

spread of each material property across the three Hall runs:
   T0         mean       273.8 K         spread   1.1 %
   N_A_cm3    mean   2.097e+12 cm^-3     spread   4.8 %
   b          mean       1.464           spread   7.9 %
   mu_H_300   mean        1104 cm^2/Vs   spread   9.1 %
   R_H_300    mean    -0.08154 m^3/C     spread  10.1 %
   rho_150

In [17]:
# one figure per panel
figs = [plt.figure(figsize=(7.5, 4.6)) for _ in range(3)]
ax = [f.add_subplot(111) for f in figs]

# rho: field dependence is real (magnetoresistance), current dependence is not
for name, c in conditions.items():
    lbl = f"{round(c['B_T']/G2T) if c['has_hall'] else 0} G, {c['I_uA']:.0f} $\\mu$A"
    ax[0].plot(c["T"], c["rho"], ".", ms=3.5, label=lbl)
ax[0].set(xlabel="T (K)", ylabel=r"$\rho$ ($\Omega\,$m)", title=r"$\rho$: all four runs")
ax[0].legend(fontsize=7.5)

for name, c in conditions.items():
    if c["has_hall"]:
        ax[1].plot(c["T"], c["R_H"], ".", ms=3.5,
                   label=f"{c['B_T']/G2T:.0f} G, {c['I_uA']:.0f} $\\mu$A")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(xlabel="T (K)", ylabel=r"$R_H$ (m$^3$/C)", title=r"$R_H$: the three Hall runs")
ax[1].legend(fontsize=7.5)

# zoom on the plateau -- where the runs should agree exactly and the noise shows
for name, c in conditions.items():
    if c["has_hall"]:
        m = (c["T"] > 100) & (c["T"] < 220)
        ax[2].plot(c["T"][m], np.abs(c["R_H"][m]), ".-", ms=4, lw=0.8,
                   label=f"{c['B_T']/G2T:.0f} G, {c['I_uA']:.0f} $\\mu$A")
ax[2].set(xlabel="T (K)", ylabel=r"$|R_H|$ (m$^3$/C)",
          title="extrinsic plateau (zoom): agreement and noise")
ax[2].legend(fontsize=7.5)

for f, nm in zip(figs, ["item13_rho_all_runs", "item13_RH_hall_runs", "item13_plateau_zoom"]):
    f.suptitle("Item 13 — comparing runs at different field and current", y=1.02)
    f.tight_layout()
    f.savefig(FIG / f"{nm}.png", bbox_inches="tight")
plt.show()

# Is the low-field condition mu*B << 1 actually satisfied? The whole simple Hall
# analysis assumes it. mu*B is exactly the ratio (applied field)/(field at which
# E_Hall = E_ohmic), i.e. the item-1 quantity turned around.
print("\nlow-field check:  mu_H * B  (the simple Hall analysis assumes this is << 1)\n")
mub = pd.DataFrame([
    dict(run=n, B_G=round(conditions[n]["B_T"] / G2T), I_uA=conditions[n]["I_uA"],
         **{f"{Tq} K": at_T(conditions[n], "mu_hall", Tq) * conditions[n]["B_T"]
            for Tq in (110, 150, 200, 300)})
    for n in conditions if conditions[n]["has_hall"]])
print(mub.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

print("\nWhat to say in the report:\n")
print(" 1. T0 agrees to 0.8 % and b to ~7 % across all three runs. Those are properties of")
print("    the germanium, not of how it was measured, and that agreement is the evidence")
print("    that the technique is sound.")
print(" 2. rho DOES depend on field, and should: that is the magnetoresistance of item 12,")
print("    not an error. Always quote rho together with the field it was measured at.")
print(" 3. The ~10 % spread in R_H and N_A is NOT random error. Look at the plateau zoom:")
print("    the curves are smooth and well separated, ordered by FIELD, with the lowest")
print("    field giving the largest |R_H|. The mu*B table above shows why -- at low")
print("    temperature mu*B approaches 0.5, so the low-field assumption is marginal. As")
print("    mu*B grows the Hall scattering factor r_H drifts from 1.18 toward 1, which")
print("    lowers the measured |R_H|. The lowest-field run is therefore the most reliable")
print("    for extracting N_A, and the field dependence itself is a result worth reporting.")
print(" 4. Current made almost no difference: compare 3000_100 (100 uA) with 3000_500")
print("    (10 uA) at nearly the same field -- R_H agrees to ~4 % and the point-to-point")
print("    scatter is the same to within a factor of 1.3. At this signal level 10 uA was")
print("    still adequate, though the manual's >=100 uA guidance remains the safe default.")


low-field check:  mu_H * B  (the simple Hall analysis assumes this is << 1)

         run  B_G  I_uA  110 K  150 K  200 K  300 K
neg_1500_100 1777   100   0.28   0.18   0.11   0.02
    3000_100 3490   100   0.49   0.29   0.19   0.04
    3000_500 3618    10   0.47   0.29   0.19   0.04
neg_1500_500 1784   500   0.28   0.18   0.12   0.02

What to say in the report:

 1. T0 agrees to 0.8 % and b to ~7 % across all three runs. Those are properties of
    the germanium, not of how it was measured, and that agreement is the evidence
    that the technique is sound.
 2. rho DOES depend on field, and should: that is the magnetoresistance of item 12,
    not an error. Always quote rho together with the field it was measured at.
 3. The ~10 % spread in R_H and N_A is NOT random error. Look at the plateau zoom:
    the curves are smooth and well separated, ordered by FIELD, with the lowest
    field giving the largest |R_H|. The mu*B table above shows why -- at low
    temperature mu*B approaches

## 12. Summary

All report quantities in one table, plus the figures written to `figures/` ready to drop into
the writeup.

In [18]:
c, mob, E = ref, M[REF], err[REF]
summary = [
    ("1", "rho at 300 K", f"{at_T(c,'rho'):.4f} +/- {at_T(c, E['sig_rho']):.4f}", "Ohm m"),
    ("1", "rho at 300 K", f"{at_T(c,'rho')*100:.2f} +/- {at_T(c, E['sig_rho'])*100:.2f}", "Ohm cm"),
    ("1", "R_H at 300 K", f"{at_T(c,'R_H'):+.4f} +/- {at_T(c, E['sig_RH']):.4f}", "m^3/C"),
    ("1", "B where E_Hall = E_ohmic", f"{1/at_T(c,'mu_hall'):.3f}  ({1/at_T(c,'mu_hall')/G2T:.0f} G)", "T"),
    ("2", "extrinsic/intrinsic transition", f"{T_rho_max:.0f}", "K"),
    ("2", "crossover region", f"{T_leave:.0f} - {T_full:.0f}", "K"),
    ("3", "band gap, ln(sigma) fit", f"{plain['Eg']:.3f} +/- {plain['err']:.3f}", "eV"),
    ("3", f"band gap, T^{ALPHA-1.5:+.2f} corrected", f"{corr['Eg']:.3f} +/- {corr['err']:.3f}", "eV"),
    ("5", "mobility ratio b, method A", f"{mob['b_A']:.2f}", ""),
    ("5", "mobility ratio b, method B", f"{mob['b_B']:.2f}", ""),
    ("6", "T_0 where R_H = 0", f"{mob['T0']:.1f}", "K"),
    ("7", "carrier type", "p-type (acceptor-doped)", ""),
    ("8", "carrier conc. at plateau", f"{mob['N_A']*1e-6:.3e}", "cm^-3"),
    ("9", "Hall mobility at 300 K", f"{at_T(c,'mu_hall')*1e4:.0f}", "cm^2/Vs"),
    ("10", "N_A, r_H = 1", f"{mob['N_A']*1e-6:.3e}", "cm^-3"),
    ("10", "N_A, r_H = 3pi/8", f"{mob['N_A_corr']*1e-6:.3e}", "cm^-3"),
    ("11", "mu_p, extrinsic", f"{mob['mu_p_ext']*1e4:.0f}", "cm^2/Vs"),
    ("11", "mu_n = b mu_p, extrinsic", f"{mob['mu_n_ext']*1e4:.0f}", "cm^2/Vs"),
    ("12", "magnetoresistance at 300 K",
     f"{100*(at_T(c,'rho')/at_T(c, c['rho_zero'])-1):+.1f}", "%"),
    ("12", "magnetoresistance at 110 K",
     f"{100*(at_T(c,'rho',110)/at_T(c, c['rho_zero'],110)-1):+.1f}", "%"),
]
tbl = pd.DataFrame(summary, columns=["item", "quantity", "value", "units"])
print(f"Summary — reference run {REF} ({c['I_uA']:.0f} uA, {c['B_T']/G2T:.0f} G)\n")
print(tbl.to_string(index=False))

tbl.to_csv("summary_results.csv", index=False)
comp.to_csv("run_comparison.csv", index=False)
print("\nwritten: summary_results.csv, run_comparison.csv")
print("figures:", *sorted(p.name for p in FIG.iterdir()), sep="\n   ")

Summary — reference run 3000_100 (100 uA, 3490 G)

item                       quantity                   value   units
   1                   rho at 300 K       0.7459 +/- 0.0090   Ohm m
   1                   rho at 300 K          74.59 +/- 0.90  Ohm cm
   1                   R_H at 300 K      -0.0901 +/- 0.0010   m^3/C
   1       B where E_Hall = E_ohmic        8.272  (82723 G)       T
   2 extrinsic/intrinsic transition                     242       K
   2               crossover region               245 - 271       K
   3        band gap, ln(sigma) fit         0.783 +/- 0.001      eV
   3    band gap, T^+0.38 corrected         0.804 +/- 0.001      eV
   5     mobility ratio b, method A                    1.55        
   5     mobility ratio b, method B                    1.58        
   6              T_0 where R_H = 0                   271.3       K
   7                   carrier type p-type (acceptor-doped)        
   8       carrier conc. at plateau               2.142e+12   cm^